# 🌲 02-机器学习 · 推导笔记 6：GBDT 梯度提升树（手写全流程）

> GBDT 是**树模型家族核心**：加法模型 + 前向分步 + 负梯度拟合。XGBoost/LightGBM 都是它的工程化。
> 本 notebook 要求：**回归树、残差拟合、学习率全部手写，不调 sklearn.ensemble**。

**运行环境**：Python 3.8+，仅依赖 numpy。

## 📖 本章导读

> **这一章讲什么**（全书第 8 章）：树模型家族的核心——**加法模型 + 前向分步 + 负梯度拟合**：每轮用一棵回归树去拟合损失函数的负梯度（伪残差），叠加时乘学习率；最后讲 XGBoost 的二阶泰勒展开与正则化。

**学完本章，你应该能**：
1. 推出平方损失下负梯度就是残差 $y-f$，二分类下是 $y-p$；
2. 解释为什么学习率 $\eta$ 是最关键的超参（步子小 → 树多 → 更稳）；
3. 说清 GBDT vs AdaBoost 的本质区别（拟合残差 vs 放大错样本）；
4. 讲清 XGBoost 的叶子权重公式 $w^*=-\frac{\sum g}{\sum h+\lambda}$ 与增益剪枝。

> 📌 全书第 8 章。阅读路径：本章导读 → 负梯度 · 学习率 · XGBoost → 自测清单 → 本章小结。

> 💡 **一句话类比——「接力纠错」**：第一个同学（第一棵树）先猜 y=5，答错了（残差 +2）；第二个同学不看原题只看「上一位差在哪」（拟合残差），补一句「应该 +2」；第三个继续纠正上一位的剩余误差……每位新手都只负责修补前面的小错误，最终拼出精准答案。这就是 GBDT：**加法模型逐轮拟合负梯度（残差）**。

## §1 加法模型与提升思想

**加法模型**：M 个弱学习器加权求和：

$$F_M(x) = \sum_{m=1}^{M} \gamma_m \, h_m(x)$$

**前向分步算法**：逐步优化，第 m 步只学当前残差：

$$\gamma_m, h_m = \arg\min_{\gamma, h} \sum_{i=1}^{n} L\big(y_i,\ F_{m-1}(x_i) + \gamma h(x_i)\big)$$

关键：**每一步在已有模型上补一个修正项**——把「短板」交给新树。

## §2 负梯度 = 伪残差（核心推导）

对平方损失 $L(y, F) = \frac{1}{2}(y - F)^2$，在函数空间做梯度下降：

$$-\frac{\partial L(y_i, F)}{\partial F} \Big|_{F=F_{m-1}(x_i)} = y_i - F_{m-1}(x_i) = \text{残差}$$

**结论**：平方损失的负梯度恰是残差 → 每棵新树拟合**当前残差**：

$$r_{im} = -\left[ \frac{\partial L(y_i, F(x_i))}{\partial F(x_i)} \right]_{F = F_{m-1}}$$

**一般化（Frechet 导数）**：任意可导损失都能定义伪残差——这就是「梯度提升」名字的由来：在函数空间对损失做梯度下降，树的拟合目标 = 负梯度。

**学习率收缩**：

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x), \quad \eta \in (0, 1]$$

小 $\eta$ + 多树 = 更平滑的拟合（正则化效果）；$\eta \cdot M \approx$ 有效容量。

In [ ]:
import numpy as np

class RegressionTree:
    """手写 CART 回归树：MSE 分裂准则、叶子取均值"""
    def __init__(self, max_depth=3, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None

    def _split_loss(self, y_left, y_right):
        # 分裂后加权 MSE（越小越好）
        def var(y):
            return np.mean((y - y.mean()) ** 2) if len(y) > 0 else 0.0
        n = len(y_left) + len(y_right)
        return (len(y_left) * var(y_left) + len(y_right) * var(y_right)) / n

    def _best_split(self, X, y):
        best = None
        best_loss = float('inf')
        n, d = X.shape
        for f in range(d):                      # 1. 遍历特征
            vals = np.unique(X[:, f])
            for v in vals[:-1]:                 # 2. 遍历分裂阈值
                mask = X[:, f] <= v
                if mask.sum() < self.min_samples_split or (~mask).sum() < self.min_samples_split:
                    continue
                loss = self._split_loss(y[mask], y[~mask])
                if loss < best_loss:            # 3. 记录最优
                    best_loss = loss
                    best = (f, v)
        return best

    def _build(self, X, y, depth):
        node = {'val': float(y.mean())}         # 叶子默认值：均值
        if depth >= self.max_depth or len(y) < 2 * self.min_samples_split:
            return node
        split = self._best_split(X, y)
        if split is None:
            return node
        f, v = split
        mask = X[:, f] <= v
        node['feature'] = f
        node['threshold'] = float(v)
        # 本次分裂的损失减少量（增益）= 父节点总损失 - 子节点加权总损失（sklearn 同款定义）
        node['gain'] = len(y) * (np.mean((y - y.mean()) ** 2) - self._split_loss(y[mask], y[~mask]))
        node['left'] = self._build(X[mask], y[mask], depth + 1)
        node['right'] = self._build(X[~mask], y[~mask], depth + 1)
        return node

    def fit(self, X, y):
        self.tree = self._build(np.asarray(X), np.asarray(y), 0)
        return self

    def _pred(self, node, x):
        if 'feature' not in node:
            return node['val']
        if x[node['feature']] <= node['threshold']:
            return self._pred(node['left'], x)
        return self._pred(node['right'], x)

    def predict(self, X):
        return np.array([self._pred(self.tree, x) for x in np.asarray(X)])

# 快速自检：单棵树拟合 y = 2*x + 噪声
rng = np.random.default_rng(0)
X1 = rng.uniform(-2, 2, (60, 1))
y1 = 2 * X1[:, 0] + rng.normal(0, 0.3, 60)
t = RegressionTree(max_depth=3).fit(X1, y1)
pred = t.predict(X1)
print('单棵树拟合 y=2x 的训练 MSE =', round(np.mean((pred - y1) ** 2), 4))
print('树的叶子预测值分布:', np.round(np.unique(pred), 3))

## §3 手写 GBDT 回归器

**算法流程**：

1. 初始化 $F_0(x) = \bar{y}$（常数模型，最优常数 = 均值）
2. 对 $m = 1 \dots M$：
   - 计算伪残差 $r_i = y_i - F_{m-1}(x_i)$
   - 回归树拟合 $\{x_i, r_i\}$ → $h_m$
   - 更新 $F_m = F_{m-1} + \eta \cdot h_m$
3. 输出 $F_M$

**每个中间量都打印**：每轮 loss、残差均值/方差（残差应逐渐趋近 0 均值、方差递减）、首棵树结构。

In [ ]:
class GBDTRegressor:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.M = n_estimators
        self.eta = learning_rate
        self.max_depth = max_depth
        self.trees = []
        self.base = None
        self.loss_history = []

    def fit(self, X, y, verbose=False):
        X = np.asarray(X)
        y = np.asarray(y).astype(float)
        self.base = float(y.mean())                      # 1. F0 = 均值
        F = np.full(len(y), self.base)
        for m in range(self.M):
            residual = y - F                             # 2. 伪残差（平方损失负梯度）
            tree = RegressionTree(max_depth=self.max_depth).fit(X, residual)
            self.trees.append(tree)
            F = F + self.eta * tree.predict(X)           # 3. 学习率收缩更新
            loss = float(np.mean((y - F) ** 2))
            self.loss_history.append(loss)
            if verbose and (m < 5 or m % 20 == 0 or m == self.M - 1):
                print(f'树{m + 1:3d}: 残差均值={residual.mean():+.4f} 残差方差={residual.var():.4f} 训练MSE={loss:.5f}')
        return self

    def predict(self, X):
        F = np.full(len(np.asarray(X)), self.base)
        for tree in self.trees:
            F = F + self.eta * tree.predict(X)
        return F

# 非线性数据：sin 曲线（树模型硬拟合非线性）
rng = np.random.default_rng(1)
X = rng.uniform(-3, 3, (200, 1))
y = np.sin(X[:, 0]) + rng.normal(0, 0.1, 200)

gbdt = GBDTRegressor(n_estimators=100, learning_rate=0.1, max_depth=3).fit(X, y, verbose=True)
print('\n最终训练 MSE =', round(gbdt.loss_history[-1], 5), '（对比：单棵树 =',
      round(np.mean((RegressionTree(max_depth=3).fit(X, y).predict(X) - y) ** 2), 5), '）')
print('loss 前3->后3:', [round(v, 4) for v in gbdt.loss_history[:3]], '->',
      [round(v, 4) for v in gbdt.loss_history[-3:]])

## §4 学习率与树数量的权衡（中间实验）

**实验**：固定 max_depth=3，改变 $\eta$，观察 loss 收敛曲线差异：

- $\eta$ 大（0.5）：收敛快但抖动、可能过拟合
- $\eta$ 小（0.05）：收敛慢但平滑

**泛化经验**：小 $\eta$ + 更多树 ≈ 更强正则；子采样（stochastic GBDT）也常用。

In [ ]:
def run_eta(eta, M=150):
    m = GBDTRegressor(n_estimators=M, learning_rate=eta, max_depth=3)
    m.fit(X, y)
    return m.loss_history

hist05 = run_eta(0.5)
hist01 = run_eta(0.1)
hist005 = run_eta(0.05)
print(' 树数 |  eta=0.5  |  eta=0.1  |  eta=0.05')
for m in [1, 5, 20, 60, 120, 149]:
    print(f'  {m:3d}  | {hist05[m]:.5f} | {hist01[m]:.5f} | {hist005[m]:.5f}')
print('观察：大学习率先降快后停滞；小学习率稳步下降（配合更多树更稳）')

## §5 特征重要性（分裂增益汇总）

**定义**：某特征在所有树中作为分裂特征带来的**MSE 下降总量**：

$$\text{importance}(f) = \sum_{\text{树 } m} \sum_{\text{含特征 } f \text{ 的分裂}} \left( \text{parent loss} - \text{child losses sum} \right)$$

归一化后即 sklearn 的 `feature_importances_`。手写版：在拟合时累计分裂增益（`gain = 父节点总损失 - 子节点加权总损失`）。**标准实现（如 sklearn）用增益累计**，而非分裂次数——分裂次数只反映"该特征被用了多少次"，不反映"它降了多少损失"。

In [ ]:
def tree_importance(X, y, n_trees=50, max_depth=3, seed=0):
    # 标准实现（如 sklearn）：按分裂带来的 MSE 减少量（增益）累计
    rng = np.random.default_rng(seed)
    n = len(y)
    imp = np.zeros(X.shape[1])
    for _ in range(n_trees):
        idx = rng.choice(n, size=int(n * 0.8), replace=False)   # bagging 子采样
        Xs, ys = X[idx], y[idx]
        F = np.full(len(idx), ys.mean())
        # 每个子样本做一轮 GBDT 更新：F0 = 均值 → 残差 → 残差树 → 累计分裂增益
        res = ys - F
        tree = RegressionTree(max_depth=max_depth).fit(Xs, res)
        walk = [tree.tree]
        while walk:
            node = walk.pop()
            if 'feature' in node:
                imp[node['feature']] += node['gain']   # 本次分裂带来的损失减少量（增益）
                walk.append(node['left'])
                walk.append(node['right'])
        F += 0.1 * tree.predict(Xs)
    return imp / imp.sum()

rng = np.random.default_rng(3)
X3 = rng.normal(0, 1, (300, 4))
y3 = 3 * X3[:, 0] - 2 * X3[:, 1] + 0.3 * X3[:, 2] + 0.1 * X3[:, 3] + rng.normal(0, 0.2, 300)
imp = tree_importance(X3, y3)
print('特征重要性（增益累计：特征0最重要，特征3最弱）:')
for f, v in enumerate(np.round(imp, 4)):
    print(f'  特征{f}: {v}')


## §6 分类 GBDT（概念 + 公式）

二分类用**对数几率**链接（类似逻辑回归）：

$$p(x) = \sigma(F(x)) = \frac{1}{1 + e^{-F(x)}}$$

损失为二分类对数损失，负梯度（伪残差）：

$$r_i = y_i - p(x_i) \quad (y_i \in \{0, 1\})$$

**叶子值修正**（不是残差均值，而是对数几率增量）：

$$\gamma_{jm} = \frac{\sum_{i \in R_{jm}} r_i}{\sum_{i \in R_{jm}} p_i(1 - p_i)}$$

树结构仍按残差拟合，只是叶子输出换为上式；多分类用 softmax 版（每类一棵树序列）。

In [ ]:
# 二分类 GBDT：负梯度 = y - p（p = sigmoid(累加输出)）；每棵树在 logit 空间拟合残差
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

class GBDTBinaryClassifier:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.M = n_estimators
        self.eta = learning_rate
        self.max_depth = max_depth
        self.trees = []
        self.base = 0.0

    def fit(self, X, y, verbose=False):
        X = np.asarray(X)
        y = np.asarray(y).astype(float)
        # F0 = 对数几率先验（p 取样本正例比例）
        p0 = y.mean()
        self.base = float(np.log((p0 + 1e-6) / (1 - p0 + 1e-6)))
        F = np.full(len(y), self.base)
        for m in range(self.M):
            p = 1 / (1 + np.exp(-F))            # sigmoid 链接到概率
            residual = y - p                     # 对数损失的负梯度 = y - p
            tree = RegressionTree(max_depth=self.max_depth).fit(X, residual)
            self.trees.append(tree)
            F += self.eta * tree.predict(X)      # logit 空间累加
            if verbose and (m < 5 or m % 20 == 0 or m == self.M - 1):
                acc = np.mean((1 / (1 + np.exp(-F)) > 0.5) == y.astype(int))
                print(f'树{m + 1:3d}: 训练准确率={acc:.4f}')
        return self

    def predict_proba(self, X):
        F = np.full(len(np.asarray(X)), self.base)
        for tree in self.trees:
            F += self.eta * tree.predict(X)
        return 1 / (1 + np.exp(-F))

    def predict(self, X):
        return (self.predict_proba(X) > 0.5).astype(int)

rng = np.random.default_rng(42)
Xc, yc = make_classification(n_samples=800, n_features=2, n_redundant=0,
                             n_informative=2, n_clusters_per_class=1,
                             class_sep=1.2, random_state=42)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.25, random_state=42)

clf = GBDTBinaryClassifier(n_estimators=100, learning_rate=0.2, max_depth=3).fit(Xc_tr, yc_tr, verbose=True)
tr_acc = np.mean(clf.predict(Xc_tr) == yc_tr)
te_acc = np.mean(clf.predict(Xc_te) == yc_te)
print(f'训练准确率 = {tr_acc:.4f}，测试准确率 = {te_acc:.4f}（>0.85 合理）')


## §7 常见 bug 与边界（面试踩坑区）

- **残差方向**：新树拟合 $y - F$ 不是 $F - y$（符号反了整个模型发散）
- **忘了学习率**：`F += h` 而非 `F += eta * h` → 过拟合快
- **分裂阈值排序**：阈值取 `np.unique` 相邻中点的经典做法此处简化为特征值本身（实现不同，效果类似）
- **min_samples_split 判断**：左右都必须满足最小样本数，漏判导致单边树
- **预测用训练均值**：predict 必须从 base 开始在**相同学习率**下累加，漏 eta 或不从 base 起都错
- **过拟合信号**：训练 MSE 持续下降但验证集回升——用小 eta + 提前停止
- **残差方差先升后降**：随机种子/噪声大时首轮残差方差可能大——看趋势而非单点

## §8 面试追问与扩展（加分项）

### 8.1 GBDT vs XGBoost vs LightGBM（必背对比）

| 维度 | GBDT | XGBoost | LightGBM |
|------|------|---------|----------|
| 分裂搜索 | 全特征全阈值 | 预排序/分位近似 | **直方图**（更快省内存） |
| 二阶导 | 只用一阶 | **用了一阶+二阶**（Newton） | 同 XGBoost |
| 正则项 | 无/弱 | 叶子数+叶子值 L2 | 同 XGBoost |
| 缺失值 | 需处理 | 自动学习方向 | 自动学习方向 |
| 加速 | — | 列块/缓存 | **GOSS + EFB**（Leaf-wise 生长） |

### 8.1.1 XGBoost 目标函数：二阶展开（牛顿步）

第 $t$ 轮的目标函数 = 损失 + 正则项：

$$\text{Obj}^{(t)} = \sum_{i=1}^{n} L\big(y_i,\ \hat{y}_i^{(t-1)} + f_t(x_i)\big) + \Omega(f_t), \qquad \Omega(f) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2$$

其中 $T$ 是叶子数，$w_j$ 是第 $j$ 个叶子的输出值。将损失 $L$ 对上一轮预测 $\hat{y}^{(t-1)}$ 做二阶泰勒展开，一阶导 $g_i$、二阶导 $h_i$ 都是对 $\hat{y}^{(t-1)}$ 求的：

$$g_i = \frac{\partial L(y_i,\ \hat{y}_i^{(t-1)})}{\partial \hat{y}^{(t-1)}}, \qquad h_i = \frac{\partial^2 L(y_i,\ \hat{y}_i^{(t-1)})}{\partial (\hat{y}^{(t-1)})^2}$$

常数项 $\sum_i L(y_i,\ \hat{y}_i^{(t-1)})$ 与本轮优化无关可略去，得到：

$$\text{Obj}^{(t)} \approx \sum_{i=1}^{n} \Big[ g_i\, w_{q(x_i)} + \tfrac{1}{2} h_i\, w_{q(x_i)}^2 \Big] + \gamma T + \frac{1}{2}\lambda \sum_{j=1}^{T} w_j^2$$

其中 $q(x_i)$ 把样本映射到其所在叶子。按叶子归组：叶子 $j$ 上的样本集 $I_j$，记 $G_j = \sum_{i \in I_j} g_i$、$H_j = \sum_{i \in I_j} h_i$，目标函数按叶子分解为 $T$ 个独立二次式：

$$\text{Obj}^{(t)} = \sum_{j=1}^{T} \Big[ G_j w_j + \tfrac{1}{2} (H_j + \lambda) w_j^2 \Big] + \gamma T$$

对每个 $w_j$ 求导并置零 → 叶子最优权重（闭合解）：

$$w_j^* = -\frac{G_j}{H_j + \lambda}$$

代回即得该树的最小目标值。分裂增益 = 分裂前（$I_L \cup I_R$）与分裂后（$I_L,\ I_R$ 两组）最小目标之差，再扣除 $\gamma$：

$$\text{Gain} = \frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} \right] - \gamma$$

**关键对比**：GBDT 只用到一阶导（负梯度 = 伪残差），每棵树拟合残差，等价于函数空间的最速下降；XGBoost 用二阶导（牛顿步），直接逼近损失函数的极小点，收敛更快，且自动按曲率 $h_i$ 加权样本。正则项 $\Omega$ 直接进目标：分裂增益必须超过 $\gamma$ 才值得分裂，$\lambda$ 收缩叶子值防过拟合——这是 XGBoost 相对 GBDT 最核心的数学差异。

### 8.1.2 XGBoost vs GBDT 关键差异对照

| 维度 | GBDT（梯度提升） | XGBoost |
|------|-----------------|---------|
| 导数阶数 | 只用一阶（负梯度 = 伪残差） | 一阶 + 二阶（牛顿步） |
| 目标函数 | 只有损失项 | 损失 + 正则（$\gamma T + \tfrac{1}{2}\lambda \sum_j w_j^2$） |
| 正则项 | 隐式（学习率/剪枝/子采样） | 显式进入目标函数 |
| 列采样 | 一般不用 | 支持（每次分裂随机选特征子集，减方差） |
| 分裂搜索 | 全特征全阈值扫描 | 预排序 + **近似直方图**（分位候选分裂点） |
| shrinkage 学习率 | 支持（$\eta$ 收缩每棵树） | 支持（$\eta$，对牛顿步同样收缩） |

### 8.2 为什么 GBDT 用树做基学习器

树对特征尺度不敏感、可处理类别特征、非线性且可解释——配合加法模型的偏差校正能力强。

### 8.3 正则化手段

学习率、子采样（行/列）、树的深度/叶子数剪枝、提前停止。

### 8.4 GBDT vs 随机森林（Bagging 对比）

RF 并行、降方差；GBDT 串行、降偏差——两者互补，组合（先 RF 后 GBDT 迭代）在 Kaggle 常用。

## §9 自测清单（L1 必会 / L2 进阶 / L3 挑战）

### L1 必会（10 问）

- [ ] 1. 写出加法模型与前向分步
- [ ] 2. 推导平方损失负梯度 = 残差
- [ ] 3. 描述 GBDT 三步循环（残差→树→更新）
- [ ] 4. 学习率的作用与默认推荐范围
- [ ] 5. 手写 CART 回归树（MSE 分裂）
- [ ] 6. 手写 GBDT 回归器（numpy）
- [ ] 7. 为什么初始化 F0 = 均值
- [ ] 8. 讲讲 feature importance 的两种定义
- [ ] 9. 手写训练 MSE 曲线输出（中间过程）
- [ ] 10. 预测阶段必须复现训练流程（base+eta）

### L2 进阶（10 问）

- [ ] 11. 分类 GBDT 叶子值公式推导
- [ ] 12. 手写分类 GBDT（对数几率 + 叶子修正）
- [ ] 13. XGBoost 目标函数推导（一阶+二阶展开 + 正则）
- [ ] 14. 手写学习率-树数权衡实验（中间表）
- [ ] 15. 子采样（stochastic GBDT）手写
- [ ] 16. 增益分裂的精确 importance 手写
- [ ] 17. 对比：GBDT vs AdaBoost（权重 vs 残差）
- [ ] 18. LightGBM 直方图加速原理
- [ ] 19. GBDT 缺失值处理策略
- [ ] 20. 手写早停（验证集 loss 监测）

### L3 挑战（5 问）

- [ ] 21. 推导 GBDT 在任意损失下的通用叶子值
- [ ] 22. 手写多分类 GBDT（softmax）
- [ ] 23. 牛顿提升（二阶）手写
- [ ] 24. XGBoost 分位近似分裂推导
- [ ] 25. GBDT 在排序学习（LambdaMART）中的应用原理

### 自测要点

- 15 分钟内盲写：回归树 + GBDT 回归器（numpy）
- 白板推导：平方损失 → 负梯度 → 残差拟合 链路
- 对比 XGBoost/LightGBM 三分钟版本

## 附录：参考与下节预告

- 《统计学习方法》第 8 章（提升方法）
- Friedman 2001：Greedy Function Approximation: A Gradient Boosting Machine

### 下节预告：推导笔记 7 — 决策树与随机森林（ID3/C4.5/CART + 信息增益手写）

> 💡 本笔记验收：`02-机器学习/README.md` 验收清单推进 1 篇。

## 🏁 本章小结

- **原理一句话**：GBDT = 每轮用回归树**拟合损失负梯度（伪残差）** + 学习率收缩。
- **必会公式**：$r_{im}=-\partial L/\partial f$（平方损失下 = $y-f$）；$F_m=F_{m-1}+\eta T_m$；XGBoost 叶子权重 $w^*=-\frac{\sum g}{\sum h+\lambda}$。
- **面试怎么考**：为什么拟合负梯度而不是残差；学习率与树数的权衡；GBDT vs AdaBoost（重残差 vs 重错样本）；XGBoost 的二阶泰勒 + 正则 + 增益剪枝。
- **易错点**：二分类负梯度是 $y-p$ 不是 $y-\hat y$；回归外的问题叶子值要线性搜索而非直接取均值。

> 自测清单没把握的条目，回到对应小节重读后再打勾。